<!-- nav -->
[← 8 · Sublanguages](08_Sublanguages.ipynb) · [Home](../../README.md) · [8.02 · Python Patterns →](08_02_Python_Patterns.ipynb)

# 8.01 · Python Types

Tutorial 8 showed each sublanguage as a dialect of its own. This page takes one, `Python Types`: what Python writes
where it means a type. A type annotation, a `TypeVar`'s bound, a type argument: the typing specification writes each as
a **type expression**, and a type expression is a Python expression, `list[int] | None`, which Python's own parser reads
as it reads any other. What Python 3.12 added beside them is not an expression: a **type parameter** of a generic
function, class or type alias, `T: int`, `*Ts` or `**P` in `def first[T: int](xs: list[T]) -> T`. `Python Types` holds
the three kinds of type parameter, and lists every kind of Python3's beside them, so that its holder holds any type
expression too ([CAP-276: Python's type expressions and match patterns as data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-276.md)).

This page builds type expressions and type parameters, from a name up, in the builders' fluent calls, prints each back
as Python writes it, validates them, finds where Python3's terms hold one, evaluates what can be evaluated, and reads the
editions that have its kinds. Like the rest of Part I, it starts from `dialects()`, a store with every dialect installed
and defined; `variable` writes a name, as tutorials 5 to 8 do. Where Python's text is an expression, Python's notebook also reads it,
with `Python3.Text.FromText`, which only Python can (tutorial 3); here each Python model is built.

In [1]:
import assert from "node:assert/strict";
import { Standards, Symbolics, Terms } from "@mbse/expressions/Framework";
import * as Python3 from "@mbse/expressions/Python/Python3";
import * as PythonTypes from "@mbse/expressions/Python/Types";
import { dialects } from "./toolkit.js";

const store = dialects();
const S = store.Expressions;
const variable = (name: string) => (d: any) => d.variable((v: any) => v.name(name));
const TYPES = PythonTypes.Expressions.DIALECT;

/** What `evaluate` says of `expression`: its value, or the reason it refuses it. */
function refused(evaluate: (expression: any) => unknown, expression: any): string {
  try {
    return String(evaluate(expression));
  } catch (error) {
    return (error as Error).message;
  }
}

<a id="T08.01.1"></a>
## 08.01.1 · A type expression is an expression

`int`, `list[int]` and `list[int] | None` are three type expressions, each a Python expression: a name, Python3's
`variable`; a subscript, Python3's `subscript`; and a union written with `|`, Python3's `binop`, whose right operand is
the singleton `None`, Python3's `constant` (tutorial 3). `Python Types`' holder, `S.Python.Types.Expression`, holds each
as its root. Its `Term` lists every branch of Python3's, so each root is a term of `Python Types`' union whose kind is
Python3's: `Terms.dialectOf` names the union that holds it, and the dialect's `owner` the dialect whose kind it is.
`Python Types` prints through Python3's printer (tutorial 8, 08.5), and Python's notebook reads each text into the same term:

In [2]:
const none = (d: any) => d.constant((k: any) => k.singleton("None").text("None")); // the singleton None, a Python3 constant
const listed = (d: any) => d.subscript((s: any) => s.value(variable("list")).slice(variable("int"))); // list[int]
const types: Record<string, any> = store.transaction(() => ({
  "int": S.Python.Types.Expression((e: any) => e.root(variable("int"))),
  "list[int]": S.Python.Types.Expression((e: any) => e.root(listed)),
  "list[int] | None": S.Python.Types.Expression((e: any) => e.root(
    (u: any) => u.binop((b: any) => b.left(listed).op("|").right(none)))),
}));
// Python reads them: `Python3.Text.FromText(source)` gives each term built here
for (const [source, expression] of Object.entries(types)) {
  const term = Terms.term(expression);
  console.log(PythonTypes.Text.ToText(expression).padEnd(17), Terms.kindOf(term)[0].padEnd(10), Terms.dialectOf(term),
    "| owner:", TYPES.owner(term).name());
  assert.equal(PythonTypes.Text.ToText(expression), source);
}
assert.equal(PythonTypes.Text.ToText, Python3.Text.ToText); // one printer for the family

int               variable   Expressions.Python.Types.Term | owner: Python3


list[int]         subscript  Expressions.Python.Types.Term | owner: Python3


list[int] | None  binop      Expressions.Python.Types.Term | owner: Python3


<a id="T08.01.2"></a>
## 08.01.2 · A type parameter

A type parameter declares a name a generic definition takes. Python writes three kinds, each `Python Types`' own, named
and shaped as mbse-programs has them (the Language Reference, 8.11, "Type parameter lists"):

```text
T                       a typevar: its name
T: int                  its bound, a type expression
T: (int, str)           its constraints, a tuple
T = str                 its default (Python 3.13)
*Ts                     a typevartuple
**P = [int, str]        a paramspec, with its default
T: list[int] | None = int
```

Each is built from the smallest up, its `name` an attribute, its `bound` and its `default_value` subterms, each a type
expression of Python3's kinds (08.01.1). A form reads a type parameter as it reads any term (tutorial 5): its kind, its
name, and its bound and default as its arguments, `None` where it has none:

In [3]:
const parameters: any[] = store.transaction(() => [
  S.Python.Types.Expression((e: any) => e.root((u: any) => u.typevar((t: any) => t.name("T")))),
  S.Python.Types.Expression((e: any) => e.root((u: any) => u.typevar((t: any) => t.name("T").bound(variable("int"))))),
  S.Python.Types.Expression((e: any) => e.root((u: any) => u.typevar((t: any) => t.name("T").bound(
    (b: any) => b.tuple((p: any) => p.elts([variable("int"), variable("str")])))))),
  S.Python.Types.Expression((e: any) => e.root((u: any) => u.typevar((t: any) => t.name("T").default_value(variable("str"))))),
  S.Python.Types.Expression((e: any) => e.root((u: any) => u.typevartuple((t: any) => t.name("Ts")))),
  S.Python.Types.Expression((e: any) => e.root((u: any) => u.paramspec((t: any) => t.name("P").default_value(
    (d: any) => d.list((p: any) => p.elements([ // a list display: the framework's list, of the framework's terms
      (t: any) => t.python3(variable("int")), (t: any) => t.python3(variable("str"))])))))),
  S.Python.Types.Expression((e: any) => e.root((u: any) => u.typevar((t: any) => t.name("T")
    .bound((b: any) => b.binop((o: any) => o.left(listed).op("|").right(none))).default_value(variable("int"))))),
]);
for (const parameter of parameters) {
  const form = Terms.form(Terms.term(parameter));
  console.log(PythonTypes.Text.ToText(parameter).padEnd(27), form.kind.padEnd(13), `name: ${String(form.attributes.get("name")).padEnd(3)}`,
    "| subterms:", form.arguments.map((a: any) => a == null ? "-" : PythonTypes.Text.ToText(a)).join(", "));
}
assert.deepEqual(parameters.map((p) => PythonTypes.Text.ToText(p)), [
  "T", "T: int", "T: (int, str)", "T = str", "*Ts", "**P = [int, str]", "T: list[int] | None = int"]);

T                           typevar       name: T   | subterms: -, -


T: int                      typevar       name: T   | subterms: int, -


T: (int, str)               typevar       name: T   | subterms: (int, str), -


T = str                     typevar       name: T   | subterms: -, str


*Ts                         typevartuple  name: Ts  | subterms: -


**P = [int, str]            paramspec     name: P   | subterms: [int, str]


T: list[int] | None = int   typevar       name: T   | subterms: list[int] | None, int


A type parameter's text is no expression, so Python's notebook reads only its bound and its default with `FromText`,
which are, and builds the parameter around them; Python's own parser reads `T: int` as a statement.

<a id="T08.01.3"></a>
## 08.01.3 · Validated, and what it binds

A type parameter declares its name for the statement that lists it, a generic `def`, `class` or `type`, which
mbse-programs models: within the term, it binds nothing. So the names its bound and its default read are references its
scope must give, its own name among them, as in `T: list[T]`; `Symbolics.free` gives them, and validation reports each
one not bound. Python3's builtins that read values, `int` and `str` among them, are always bound (tutorial 4); `list` is
not one, so the scope gives it. Validation checks each kind's own rules too, with its own message: a type parameter's
name must be an identifier, so `1T` is kept as written, printed, and refused
([PRI-002: preserve, don't lose](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-002)):

In [4]:
const [recursive, misnamed]: any[] = store.transaction(() => [
  S.Python.Types.Expression((e: any) => e.root((u: any) => u.typevar((t: any) => t.name("T").bound(
    (b: any) => b.subscript((s: any) => s.value(variable("list")).slice(variable("T"))))))),
  S.Python.Types.Expression((e: any) => e.root((u: any) => u.typevar((t: any) => t.name("1T")))),
]);
for (const expression of [parameters.at(-1), recursive, misnamed]) {
  console.log(PythonTypes.Text.ToText(expression).padEnd(27), "free:", [...Symbolics.free(expression, TYPES)].sort().join(", ") || "none",
    "|", TYPES.validate(expression).join("; ") || "valid");
}
assert.deepEqual([...TYPES.validate(parameters.at(-1), ["list"]), ...TYPES.validate(recursive, ["list", "T"])], []); // its scope gives them
assert.deepEqual(TYPES.validate(misnamed), ["a typevar's name must be an identifier, got '1T'"]);

T: list[int] | None = int   free: list | bound: left: value: variable 'list' is not bound


T: list[T]                  free: T, list | bound: value: variable 'list' is not bound; bound: slice: variable 'T' is not bound


1T                          free: none | a typevar's name must be an identifier, got '1T'


<a id="T08.01.4"></a>
## 08.01.4 · Where Python3 holds one

Python writes a type expression where it annotates: a parameter's annotation, a function's return type, a variable's.
Those are statements and definitions, mbse-programs', but a parameter is a helper Python3 shares with them, `arg`
(tutorial 6), whose `annotation` is a subterm of `Python Types`' `Term`. So Python3 reads `Python Types`' terms, as
each dialect of the family reads the others', `reads()`. The one expression with parameters, `lambda`, takes no
annotations in Python's grammar: built with one, it is held as written, and validation says what Python's parser would:

In [5]:
const annotated = store.transaction(() => S.Python.Python3.Expression((e: any) => e.root((u: any) => u.lambda((l: any) => l
  .args((a: any) => a.args([(g: any) => g.arg("x").annotation((t: any) => t.binop(
    (b: any) => b.left(variable("int")).op("|").right(none)))]))
  .body(variable("x"))))));
const annotation = Terms.form(Terms.term(annotated)).arguments[2]; // after Let's empty value and its body: x's annotation
console.log("Python3 reads", Python3.Expressions.DIALECT.reads().map((d: any) => d.name()).join(", "));
console.log(Python3.Text.ToText(annotation).padEnd(10), Terms.dialectOf(annotation), "| owner:",
  Python3.Expressions.DIALECT.owner(annotation).name());
console.log(...Python3.Expressions.DIALECT.validate(annotated));

Python3 reads Python Types, Python Patterns, Python RegExp


int | None Expressions.Python.Types.Term | owner: Python3


a lambda's parameters have no annotations


<a id="T08.01.5"></a>
## 08.01.5 · Evaluated as Python3's, or refused by name

A type expression is evaluated as the Python3 expression it is, by Python3's evaluator, in a scope that gives its names:
`int if wide else float` chooses a type by a flag. A type parameter is not evaluated: a statement declares it, and
Python3's evaluator refuses each of `Python Types`' kinds by the dialect's name (tutorial 8, 08.3):

In [6]:
const chosen = store.transaction(() => S.Python.Types.Expression((e: any) => e.root((u: any) => u.ifexp((i: any) => i
  .test(variable("wide")).body(variable("int")).orelse(variable("float"))))));
const scope = { wide: true, int: "int64", float: "float32" };
console.log(PythonTypes.Text.ToText(chosen).padEnd(24), Python3.Evaluators.OfAny(chosen, scope));
for (const parameter of [parameters[0], parameters[4], parameters[5]]) {
  console.log(PythonTypes.Text.ToText(parameter).padEnd(24), refused((x) => Python3.Evaluators.OfAny(x), parameter));
}

int if wide else float   int64


T                        Python Types typevar is not evaluated


*Ts                      Python Types typevartuple is not evaluated


**P = [int, str]         Python Types paramspec is not evaluated


<a id="T08.01.6"></a>
## 08.01.6 · Its editions as data

`Python Types` covers the versions of Python that Python3 covers, 3.0 to 3.15, each found by its published name
([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)),
and records which of its own kinds each has: none before Python 3.12, which added type parameter lists, all three
since. The kinds it lists from Python3 are Python3's, recorded there (tutorial 2, 02.1):

In [7]:
const dialect = store.Instances.Expressions.Python.Types;
const editions = Standards.standards(dialect);
console.log(editions.size, "editions:", [...editions.keys()].slice(0, 2).join(", "), "...", [...editions.keys()].at(-1));
for (const version of ["Python 3.11", "Python 3.12", "Python 3.15"]) {
  console.log(version.padEnd(12), Standards.kinds(Standards.find(dialect, version)!).map((kind: any) => kind.name.split(".").at(-1)).join(", ")
    || "none of its own");
}

16 editions: Python 3.0, Python 3.1 ... Python 3.15


Python 3.11  none of its own


Python 3.12  TypeVar, ParamSpec, TypeVarTuple


Python 3.15  TypeVar, ParamSpec, TypeVarTuple


## What you learned

- **A type expression is a Python expression**, of Python3's kinds, held by `Python Types`' holder, whose `Term` lists
  every branch of Python3's; Python reads its text and prints it back.
- **A type parameter is `Python Types`' own**: `typevar`, `typevartuple` and `paramspec`, a name with a bound and a
  default, each a type expression.
- **It binds nothing within a term**: the statement that lists it binds its name, so the names its bound and default
  read are references, which validation reports where no scope gives them.
- **Python3 holds a type expression** where a parameter is annotated, and reads `Python Types`' terms there.
- **A type expression evaluates as Python3's; a type parameter is refused** by its dialect's name.
- **Its editions are data**: Python 3.0 to 3.15, its own kinds since 3.12.

Next, `Python Patterns`: the patterns of Python's `match` statement.

<!-- nav -->
[← 8 · Sublanguages](08_Sublanguages.ipynb) · [Home](../../README.md) · [8.02 · Python Patterns →](08_02_Python_Patterns.ipynb)